# 03 · Retrieving 850 hPa wind for the selected days

**Goal.** Fetch 850 hPa horizontal wind (`u`, `v`) for exactly the days chosen in notebook 02,
and understand why the *access pattern* — not the data volume — drives the cost.

## Why a separate retrieval

The `era5-monthly-pl-nwsa.zarr` store from notebook 01 has 850 hPa `U`/`V`, but as **monthly
means**. A monthly mean wind field cannot classify individual days: averaging a month of
synoptic variability erases exactly the day-to-day flow differences the SOM is meant to find.

So daily wind has to come from elsewhere. We use **ARCO-ERA5** — Google's public,
analysis-ready Zarr copy of ERA5, hourly, global, 0.25°, all 37 pressure levels:

```
gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3
```

It requires no credentials (anonymous GCS read) and no request queue, which makes it
substantially more convenient than ordering the equivalent subset from a reanalysis portal.

## The chunking argument — the key idea in this notebook

ARCO-ERA5 is chunked **one hour per chunk**, with each chunk holding a *full global field across
all 37 levels* for a single variable. That geometry decides everything:

| Access pattern | Chunks touched | Verdict |
|---|---|---|
| A few hundred **scattered** days, one hour each | 1 per day per variable | **cheap** — our case |
| A multi-decade **continuous** hourly record | one per hour, millions | expensive |
| One grid point, long time series | one per hour, millions | pathologically expensive |

Our day list is ~100 non-contiguous dates spanning 40+ years. That is the *best* case for this
layout: we read one chunk per day and throw away everything outside the NWSA window and off the
850 hPa level. The wasted bytes per chunk are large, but the **number of chunk reads is tiny**,
and chunk count is what dominates wall time for object-store access.

The corollary: if you adapt this notebook to want a continuous multi-year daily record, this
store stops being the right choice and a per-day-bundled archive becomes faster.

## One synoptic snapshot vs. a daily mean

We take a single **12:00 UTC** instant (≈07:00 local in Ecuador) rather than averaging
00/06/12/18 UTC.

- **Cost:** a daily mean is 4× the chunk reads, for the same final field size.
- **Science:** a single synoptic snapshot is the standard input for circulation-typing studies
  and preserves the instantaneous large-scale flow. A daily mean smooths the diurnal cycle,
  which is *desirable* if you want the day's representative state and *undesirable* if diurnal
  timing is part of the signal.

For classifying large-scale flow regimes the snapshot is adequate and 4× cheaper. Set
`HOURS = [0, 6, 12, 18]` below to switch — the rest of the chain is unchanged.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import sys
sys.path.append("../src")
from config import (ARCO_STORE, DATA_DIR, NWSA_LAT, NWSA_LON_360,
                    WIND_LEVEL, SNAPSHOT_HOUR, open_arco)

days = pd.read_csv(DATA_DIR / "heavy_enso_positive_days.csv", parse_dates=["date"])
print(len(days), "days;", days.date.min().date(), "->", days.date.max().date())
print("store:", ARCO_STORE)
print("level:", WIND_LEVEL, "hPa;  snapshot hour:", SNAPSHOT_HOUR, "UTC")

137 days; 1983-01-16 -> 2026-03-12
store: gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3
level: 850 hPa;  snapshot hour: 12 UTC


## 1. Inspect the store before pulling from it

Two things must be confirmed before writing the retrieval loop, because guessing either costs a
failed multi-minute job:

- **Variable names.** ARCO-ERA5 uses long CF-style names (`u_component_of_wind`), not the short
  GRIB names (`u`) that a CDS-downloaded netCDF would have.
- **Coordinate conventions.** Longitude is **0–360**, not −180–180. The NWSA window
  `lon [-95, -60]` becomes `lon [265, 300]`. Latitude **descends** (north to south), so slices
  must be given high-to-low.

In [2]:
# open_arco() prefers gs:// via gcsfs and falls back to the bucket-qualified HTTPS
# endpoint where the path-style GCS API is blocked -- see src/config.py
ds = open_arco()

wanted = ["u_component_of_wind", "v_component_of_wind", "specific_humidity"]
print("target variables present:", {v: v in ds.data_vars for v in wanted})
print("\npressure levels (hPa):")
print(ds.level.values)
print("\nlongitude convention:", float(ds.longitude.min()), "to", float(ds.longitude.max()))
print("latitude order:", float(ds.latitude.values[0]), "->", float(ds.latitude.values[-1]),
      "(descending)" if ds.latitude.values[0] > ds.latitude.values[-1] else "(ascending)")
print("\nchunking of u_component_of_wind:", ds["u_component_of_wind"].encoding.get("chunks"))

target variables present: {'u_component_of_wind': True, 'v_component_of_wind': True, 'specific_humidity': True}

pressure levels (hPa):
[   1    2    3    5    7   10   20   30   50   70  100  125  150  175
  200  225  250  300  350  400  450  500  550  600  650  700  750  775
  800  825  850  875  900  925  950  975 1000]

longitude convention: 0.0 to 359.75
latitude order: 90.0 -> -90.0 (descending)

chunking of u_component_of_wind: (1, 37, 721, 1440)


## 2. The retrieval loop

One `.sel(...).load()` per day. Deliberately a plain Python loop rather than a single
vectorized `.sel(time=[...])`:

- Each day is an independent chunk read, so there is nothing to gain from batching.
- A per-day loop lets a single missing or corrupt timestep be **skipped and reported** rather
  than failing the whole multi-minute job — worth a lot when the job runs unattended.
- Progress is visible, so a stalled read is obvious.

Note `slice(NWSA_LAT[1], NWSA_LAT[0])` — high latitude first, matching the descending axis.

In [3]:
def fetch_wind_snapshots(ds, dates, level=WIND_LEVEL, hour=SNAPSHOT_HOUR):
    """850 hPa u,v on the NWSA window for each date, at a fixed UTC hour.

    Returns (u_stack, v_stack, used_dates, lat, lon) -- longitudes still 0-360.
    """
    lat_ref = ds.latitude.sel(latitude=slice(NWSA_LAT[1], NWSA_LAT[0])).values   # descending
    lon_ref = ds.longitude.sel(longitude=slice(*NWSA_LON_360)).values

    u_list, v_list, used = [], [], []
    for i, d in enumerate(dates):
        t = pd.Timestamp(d).normalize() + pd.Timedelta(hours=hour)
        try:
            sel = ds[["u_component_of_wind", "v_component_of_wind"]].sel(
                time=t, level=level,
                latitude=slice(NWSA_LAT[1], NWSA_LAT[0]),
                longitude=slice(*NWSA_LON_360),
            ).load()
        except Exception as exc:
            print(f"  SKIP {t}: {type(exc).__name__}: {exc}")
            continue
        u_list.append(sel["u_component_of_wind"].values)
        v_list.append(sel["v_component_of_wind"].values)
        used.append(pd.Timestamp(d).normalize())
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(dates)}", flush=True)

    return np.stack(u_list), np.stack(v_list), used, lat_ref, lon_ref

In [4]:
# Roughly a minute per few dozen days over a good connection. Cached afterwards.
wind_path = DATA_DIR / "era5_850_wind_heavy_enso.nc"

if wind_path.exists():
    wind = xr.open_dataset(wind_path)
    print("loaded cached wind file")
else:
    U, V, used, lat_ref, lon_ref = fetch_wind_snapshots(ds, days["date"].values)
    wind = xr.Dataset(
        {"u850": (("time", "latitude", "longitude"), U),
         "v850": (("time", "latitude", "longitude"), V)},
        coords={"time": pd.DatetimeIndex(used), "latitude": lat_ref, "longitude": lon_ref},
    )
    wind.attrs.update({
        "source": ARCO_STORE,
        "level_hPa": WIND_LEVEL,
        "time_of_day": f"{SNAPSHOT_HOUR:02d}:00 UTC snapshot",
        "n_days": len(used),
        "note": "850 hPa wind for heavy El Nino precip days over Guayas; SOM input",
    })
    wind.to_netcdf(wind_path)
    print("wrote", wind_path)

print(wind.sizes)

  20/137
  40/137
  60/137
  80/137
  100/137
  120/137
wrote /Users/snesbitt/.claude-science/orgs/6a2220b0-49b1-45ab-a891-23ce4097f835/workspaces/d6236080-0839-49c3-a3ca-b04e592b5786/ecuador-enso-som/data/era5_850_wind_heavy_enso.nc
Frozen({'time': 137, 'latitude': 141, 'longitude': 141})


## 3. Verify the retrieval

Checks that catch the failure modes that actually occur:

- **Completeness** — did every requested day come back?
- **Physical plausibility** — 850 hPa winds of order 0–30 m/s; a field of zeros or 1e20 means a
  fill value or a wrong level index.
- **No all-NaN fields** — a mis-specified slice returns an empty or NaN array without raising.

In [5]:
got = pd.to_datetime(wind.time.values).normalize()
want = pd.to_datetime(days["date"].values)

print("requested :", len(want))
print("retrieved :", len(got))
print("missing   :", sorted(set(want) - set(got)))
print()
print("u850 range: %.2f to %.2f m/s" % (float(wind.u850.min()), float(wind.u850.max())))
print("v850 range: %.2f to %.2f m/s" % (float(wind.v850.min()), float(wind.v850.max())))
print("grid      :", wind.sizes["latitude"], "x", wind.sizes["longitude"])
print("non-finite fraction:", float((~np.isfinite(wind.u850)).mean()))

spd = np.sqrt(wind.u850.values**2 + wind.v850.values**2)
print("wind speed: mean %.2f, 99th pct %.2f, max %.2f m/s"
      % (spd.mean(), np.percentile(spd, 99), spd.max()))

requested : 137
retrieved : 137
missing   : []

u850 range: -23.07 to 20.85 m/s
v850 range: -29.76 to 26.96 m/s
grid      : 141 x 141
non-finite fraction: 0.0
wind speed: mean 5.26, 99th pct 15.59, max 31.37 m/s


## 4. A data-quality issue you must handle before interpreting anything

**The 850 hPa surface intersects the Andes.**

850 hPa corresponds to roughly 1500 m altitude. Over terrain higher than that, the 850 hPa
"level" lies *at or below ground*. ERA5 still reports a value there, but it is a downward
extrapolation of the model's free-atmosphere state, not a measurement of real wind.

This is not a small correction in a corner of the domain — the Andes run north–south straight
through the middle of it. Untreated, it produces a striking band of near-zero wind along the
cordillera in *every* composite, which is easy to mistake for a real orographic wind shadow.

The diagnostic below is the one that settles it: stratify wind speed by terrain elevation. A
physical signal would vary between flow regimes; an extrapolation artifact collapses to the same
near-zero values in **every** case.

In [6]:
topo = xr.open_dataarray(DATA_DIR / "etopo2_nwsa.nc")
lon180 = wind.longitude.values - 360         # back to -180..180 to match the topo file
elev = topo.interp(lat=("latitude", wind.latitude.values),
                   lon=("longitude", lon180)).values

bands = [(-1e9, 0, "ocean"), (0, 500, "0-500 m"), (500, 1500, "500-1500 m"),
         (1500, 3000, "1500-3000 m"), (3000, 1e9, ">3000 m")]

rows = []
for lo, hi, lab in bands:
    m = (elev > lo) & (elev <= hi)
    if m.sum():
        rows.append({"band": lab, "n_cells": int(m.sum()),
                     "mean_speed_ms": round(float(spd[:, m].mean()), 2)})
print(pd.DataFrame(rows).to_string(index=False))

       band  n_cells  mean_speed_ms
      ocean    10698           5.27
    0-500 m     6616           6.28
 500-1500 m     1229           4.05
1500-3000 m      608           1.32
    >3000 m      730           1.07


The monotonic collapse with elevation is the signature of the artifact. Every figure in
notebooks 04 and 05 therefore **masks terrain above 1500 m** and says so in its caption.

Two things worth being clear about:

- The mask is a *presentation* decision. The SOM in notebook 04 is trained on the **unmasked**
  field, because masking would introduce NaNs into the feature vectors. The high-terrain cells
  contribute (mostly uninformative, near-constant) values to the distance metric rather than
  being excluded — a limitation to note, not a bug.
- If you need wind that is physically meaningful *over* the Andes, 850 hPa is the wrong level.
  Use 700 hPa (≈3000 m) or a terrain-following coordinate.

In [7]:
TERRAIN_MASK_M = 1500
terrain_mask = elev > TERRAIN_MASK_M

np.save(DATA_DIR / "terrain_mask_1500m.npy", terrain_mask)
print(f"masked cells (> {TERRAIN_MASK_M} m): {int(terrain_mask.sum())} "
      f"of {terrain_mask.size} ({100*terrain_mask.mean():.1f}%)")

masked cells (> 1500 m): 1338 of 19881 (6.7%)


## What notebook 04 needs from here

- `data/era5_850_wind_heavy_enso.nc` — `u850`, `v850` on `(time, latitude, longitude)`
- `data/terrain_mask_1500m.npy` — boolean mask, `True` where 850 hPa is unreliable

Notebook 05 reuses the same store and the same access pattern to pull humidity and wind on 20
levels for the vapor-transport calculation.